# Project 5 — Inventory and Supply Chain Analysis

Analyze demand, fulfillment, stockouts, replenishment, and potential inventory-policy improvements. The monthly product-warehouse data are synthetic and designed for practice.

## Load inventory and product tables

The inventory table is at product × warehouse × month grain. A repeated product ID is expected because products appear across warehouses and months. Joining the product table adds descriptive attributes; avoid creating duplicate rows with a many-to-many join.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path  # Import utilities for resolving local data paths.
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")  # Support running from the project root or notebooks folder.
inventory = pd.read_csv(DATA_DIR / "inventory_monthly.csv", parse_dates=["month"])  # Load monthly inventory facts.
products = pd.read_csv(DATA_DIR / "products.csv")  # Load product attributes.
df = inventory.merge(products[["product_id", "category", "supplier_id"]], on="product_id", how="left", validate="many_to_one")  # Join product details safely.
print("Inventory rows:", len(inventory), "| merged rows:", len(df))  # Check that the join did not multiply rows.
print(df.head())  # Preview inventory observations.
print("\nMissing values:\n", df.isna().sum())  # Check whether the join or input data left missing values.

## Calculate fulfillment and stockout KPIs

Service and stock KPIs help quantify both customer impact and working capital. Fulfillment rate here is units fulfilled divided by demand units. A stockout rate at the product-warehouse-month level is the share of rows with any unmet demand.

These definitions are not universal; confirm how partial fulfillment, backorders, and canceled orders are handled in the real system.

In [ ]:
total_demand = df["demand_units"].sum()  # Sum demand units across all product-warehouse-month observations.
total_fulfilled = df["fulfilled_units"].sum()  # Sum fulfilled units.
total_stockout_units = df["stockout_units"].sum()  # Sum units of unmet demand.
fulfillment_rate = total_fulfilled / total_demand if total_demand else np.nan  # Calculate units fulfilled as a share of demand.
stockout_row_rate = (df["stockout_units"] > 0).mean()  # Calculate the share of product-warehouse-month rows with any stockout.
stockout_value = df["estimated_stockout_value"].sum()  # Estimate demand value associated with unmet units.
print(f"Demand units: {total_demand:,}")  # Display total demand.
print(f"Fulfilled units: {total_fulfilled:,}")  # Display units fulfilled.
print(f"Unit fulfillment rate: {fulfillment_rate:.1%}")  # Display overall fulfillment.
print(f"Product-warehouse-month stockout rate: {stockout_row_rate:.1%}")  # Display stockout-row share.
print(f"Estimated stockout value: {stockout_value:,.2f}")  # Display a proxy for value of unmet demand.

## Find stockout hotspots

Rank warehouses, categories, and products using unmet units, the share of rows with stockouts, and estimated stockout value. High-volume items can matter even when their stockout rate is moderate; low-volume items can have a high rate but limited impact.

Prioritize using both customer impact and financial impact, then validate causes such as demand spikes, supplier delays, order policy, or forecast error.

In [ ]:
category_stockouts = df.groupby("category").agg(  # Summarize stockout patterns by category.
    demand_units=("demand_units", "sum"),  # Sum demand.
    stockout_units=("stockout_units", "sum"),  # Sum unmet demand.
    stockout_value=("estimated_stockout_value", "sum"),  # Sum estimated value associated with unmet demand.
    product_months=("stockout_units", "size"),  # Count product-warehouse-month observations.
    rows_with_stockout=("stockout_units", lambda s: (s > 0).sum()),  # Count observations with any unmet demand.
)  # Finish category aggregation.
category_stockouts["stockout_row_rate"] = category_stockouts["rows_with_stockout"] / category_stockouts["product_months"]  # Calculate stockout-row rate.
category_stockouts["unit_fill_rate"] = (category_stockouts["demand_units"] - category_stockouts["stockout_units"]) / category_stockouts["demand_units"]  # Calculate unit fill rate.
print(category_stockouts.sort_values("stockout_value", ascending=False).round(3))  # Rank categories by estimated stockout value.

## Inspect inventory trends

Monthly analysis helps separate one-off spikes from recurring stock problems. Track demand, stockouts, closing stock, and replenishments together; otherwise, reducing stockouts by holding excessive inventory could look like an improvement.

Inventory turnover usually requires a consistent cost-of-goods-sold and average-inventory valuation. This practice dataset allows operational indicators, but a production financial turnover KPI needs finance-approved definitions.

In [ ]:
monthly = df.groupby("month").agg(  # Aggregate inventory performance by month.
    demand_units=("demand_units", "sum"),  # Sum demand units each month.
    stockout_units=("stockout_units", "sum"),  # Sum unmet demand units.
    closing_stock=("closing_stock", "sum"),  # Sum ending inventory across tracked locations and products.
    replenishment_received=("replenishment_received_units", "sum"),  # Sum received replenishment units.
)  # Finish the monthly trend table.
monthly["unit_fill_rate"] = (monthly["demand_units"] - monthly["stockout_units"]) / monthly["demand_units"]  # Calculate monthly fulfillment rate.
fig, ax = plt.subplots(figsize=(9, 4))  # Create a chart for trend comparison.
monthly[["stockout_units", "replenishment_received"]].plot(ax=ax, marker="o")  # Plot stockout units and replenishment units over time.
ax.set_title("Monthly Stockouts and Replenishments")  # Add the chart title.
ax.set_xlabel("Month")  # Label time.
ax.set_ylabel("Units")  # Label quantities.
plt.tight_layout()  # Adjust plot spacing.
plt.show()  # Display the chart.
print(monthly.round(2))  # Display the data behind the chart.

## Build a replenishment review

A reorder point is often based on demand during lead time plus safety stock. For stable daily demand, a basic formula is `average daily demand × lead time + safety stock`. More sophisticated methods account for demand and lead-time variability.

The supplied reorder-point field is illustrative. Compare current policy with observed stockouts, supplier lead times, and average inventory; do not change reorder settings without validating replenishment timing and operational constraints.

In [ ]:
product_policy = df.groupby(["product_id", "category"]).agg(  # Aggregate a product's policy and outcomes across locations and months.
    mean_monthly_demand=("demand_units", "mean"),  # Estimate average monthly demand.
    mean_lead_days=("supplier_lead_days", "mean"),  # Estimate average supplier lead time.
    mean_reorder_point=("reorder_point_units", "mean"),  # Summarize the illustrative reorder point.
    stockout_units=("stockout_units", "sum"),  # Sum total stockout units.
    mean_closing_stock=("closing_stock", "mean"),  # Estimate average ending stock.
)  # Finish product policy aggregation.
product_policy["stockout_units_per_100_demand"] = 100 * product_policy["stockout_units"] / (product_policy["mean_monthly_demand"] * 18 * 4)  # Estimate a normalized stockout indicator for 18 months and four warehouses.
print(product_policy.sort_values("stockout_units", ascending=False).head(12).round(2))  # Show products with the most unmet demand.
print("Review high stockout products with supplier constraints, demand volatility, and inventory holding cost before acting.")  # Add a decision caveat.